# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook guides you through loading and exploring the [FAIR^2 tabular dataset](https://sen.science/doi/10.71728/senscience.qs2f-h81p) using the `mlcroissant` library. The dataset's Croissant schema is referenced via a publicly accessible URL and includes structured record sets and fields suitable for machine-learning workflows.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
```
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
```

Let's get started!

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL (the schema in Croissant JSON-LD format)
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}\n")
print(f"Published: {metadata.datePublished}, Identifier: {metadata.identifier}")
print(f"Version: {metadata.version}")
print(f"License: {metadata.license}")

## 2. Data Overview

Review available record sets, fields, and their `@id`s.

We will inspect the dataset's record sets and fields using their unique `@id` attributes, as recommended by the Croissant specification.

In [ ]:
print("Available record sets in the dataset:")

record_sets = [r for r in metadata.recordSets]
for rs in record_sets:
    print(f"- Record Set: @id={rs['@id']}")
    print(f"  name: {rs.get('name', 'N/A')}")
    print(f"  description: {rs.get('description', 'N/A')}")
    # List fields in this recordSet
    if 'fields' in rs:
        print("  Fields:")
        for fld in rs['fields']:
            print(f"    - Field: @id={fld['@id']} | name={fld.get('name', 'N/A')} | dataType={fld.get('dataType', 'N/A')}")
    print('')

# For illustration, make a list of just the record set @ids
record_set_ids = [rs['@id'] for rs in record_sets]
print(f"Record set @ids available: {record_set_ids}")

# If the dataset only has one record set, display the first 3 records
if record_set_ids:
    print("\nSample records from first Record Set:")
    for i, rec in enumerate(dataset.records(record_set=record_set_ids[0])):
        if i >= 3: break
        print(rec)

## 3. Data Extraction

Load data from a specific record set into a DataFrame for analysis.

We will select the first record set by its `@id`, and show column information by `@id` as defined in the schema.

In [ ]:
# Extract data for all record sets in the schema.

# record_set_ids is already defined in previous cell
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading records for record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f" - Columns (@id): {list(df.columns)}")

# Preview the data from the primary record set
primary_record_set = record_set_ids[0]
print(f"\nFirst few rows of DataFrame for record set '{primary_record_set}':")
display(dataframes[primary_record_set].head())

## 4. Exploratory Data Analysis (EDA)

We'll pick a relevant numeric field for demonstration, apply filtering and normalization, and show grouping by a categorical field. We always reference fields/columns by their Croissant `@id`.

In [ ]:
# Pick a numeric field for processing, referencing it by @id

# Let's find a likely numeric field among the columns
df = dataframes[primary_record_set]
numeric_candidate = None
group_candidate = None

print("\nFields and column names (by @id):")
for col in df.columns:
    print(f" - {col}")
    # Guess by common field name - replace with actual field @id if known
    if ('age' in col.lower() or 'interval' in col.lower() or 'years' in col.lower()) and numeric_candidate is None:
        numeric_candidate = col
    if 'sex' in col.lower() or 'gender' in col.lower() or 'location' in col.lower() or 'site' in col.lower():
        group_candidate = col

# Fallbacks: just the first numeric-looking column
if not numeric_candidate:
    # Try to find float/int columns
    for c in df.columns:
        if pd.api.types.is_numeric_dtype(df[c]):
            numeric_candidate = c
            break
if not numeric_candidate:
    # Fallback: just use first column
    numeric_candidate = df.columns[0]

if not group_candidate:
    group_candidate = df.columns[-1]

print(f"\nSelected numeric field for EDA (by @id): {numeric_candidate}")
print(f"Selected grouping field for EDA (by @id): {group_candidate}")

# Ensure the column is numeric
df[numeric_candidate] = pd.to_numeric(df[numeric_candidate], errors='coerce')

threshold = df[numeric_candidate].mean() if not df[numeric_candidate].isnull().all() else 0

filtered_df = df[df[numeric_candidate] > threshold]

print(f"Filtered records where {numeric_candidate} > {threshold:.2f}:")
display(filtered_df.head())

# Normalize the numeric field
filtered_df[f"{numeric_candidate}_normalized"] = (
    (filtered_df[numeric_candidate] - filtered_df[numeric_candidate].mean()) /
    filtered_df[numeric_candidate].std()
)

print(f"Normalized {numeric_candidate} for filtered records:")
display(filtered_df[[numeric_candidate, f"{numeric_candidate}_normalized"]].head())

# Grouping
if group_candidate in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_candidate)[numeric_candidate].mean()
    print(f"\nGrouped mean of {numeric_candidate} by {group_candidate}:")
    display(grouped_df.head())

## 5. Visualization

Let's visualize the distribution of the selected numeric field and its relationship to a categorical/grouping variable.

All field references in plots are by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(7,4))
sns.histplot(df[numeric_candidate].dropna(), kde=True)
plt.title(f"Distribution of {numeric_candidate}")
plt.xlabel(numeric_candidate)
plt.ylabel("Count")
plt.show()

# Boxplot of numeric field grouped by group_candidate
if group_candidate in df.columns:
    plt.figure(figsize=(8,4))
    sns.boxplot(x=df[group_candidate], y=df[numeric_candidate])
    plt.title(f"{numeric_candidate} by {group_candidate}")
    plt.xlabel(group_candidate)
    plt.ylabel(numeric_candidate)
    plt.show()

## 6. Conclusion

- We successfully loaded the FAIR^2 dataset using its Croissant schema and explored its structure using `mlcroissant`.
- Data was accessed and manipulated by unique `@id` references for all record sets and fields, ensuring robust and reproducible operations.
- Basic EDA demonstrated numeric field normalization and grouping, and we visualized field distributions.
- For further insights, consider exploring more complex relationships, leveraging the schema metadata for data quality checks, or applying ML algorithms with confidence in data provenance.

For more details on the dataset, visit [FAIR^2 Dataset landing page](https://sen.science/doi/10.71728/senscience.qs2f-h81p) or consult the Croissant [specification](https://mlcommons.org/croissant) and [mlcroissant documentation](https://github.com/mlcommons/croissant).